## Import

In [8]:
import os
import mne
from mne import events_from_annotations, create_info, EpochsArray, concatenate_epochs, Epochs
import json
import pandas as pd
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import warnings
from collections import defaultdict
from eegkit.models import (
    TaskDTO, FilterParamsDTO, TimeDomainParamsDTO, PSDParamsDTO,
    EpochParamsDTO, EpochFullParamsDTO, TableInfoDTO, EpochPSDParamsDTO
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

from notebook_utils import reload_classes, reload_data_classes

warnings.filterwarnings("ignore", message=".*boundary.*data discontinuities.*")
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

## Load

In [9]:
confirmation = input("Type '1' to reload: ")

if confirmation.lower() == '1':
    EEGSubjectData = reload_data_classes()
    release = 1
    data_dir = f'/mount/intern/NAS-public-dataset/HBN-EEG/cmi_bids_R{release}'
    subject_data = EEGSubjectData(data_dir)

In [10]:
EEGController, EEGUI = reload_classes()
controller = EEGController(subject_data)
subject_model = controller.subject_model
visualizer = controller.visualizer
data_service = controller.data_service

In [11]:
subjects = sorted(controller.list_subjects())
subject = subjects[0]
subject

'sub-NDARAC904DMU'

In [12]:
task_keys = sorted(controller.list_tasks(subject))
for (i, item) in enumerate(task_keys):
    print(i, item)

0 ('DespicableMe', None)
1 ('DiaryOfAWimpyKid', None)
2 ('FunwithFractals', None)
3 ('RestingState', None)
4 ('ThePresent', None)
5 ('contrastChangeDetection', '1')
6 ('contrastChangeDetection', '2')
7 ('contrastChangeDetection', '3')
8 ('seqLearning8target', None)
9 ('surroundSupp', '1')
10 ('surroundSupp', '2')
11 ('symbolSearch', None)


# NN Class

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [14]:
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

# List all devices with their names
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Current default device index
if torch.cuda.is_available():
    print(f"Current default device: cuda:{torch.cuda.current_device()}")
else:
    print("No CUDA-compatible GPU found.")

Number of GPUs: 0
No CUDA-compatible GPU found.


In [15]:
import torch, platform, sys
print("PyTorch:", torch.__version__)
print("CUDA build in torch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Python:", sys.version.split()[0], "OS:", platform.platform())

PyTorch: 2.4.1
CUDA build in torch: None
CUDA available: False
GPU count: 0
Python: 3.12.11 OS: Linux-5.15.0-46-generic-x86_64-with-glibc2.31


In [16]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

class CNNLSTMDense(nn.Module):
    """
    Input:  x of shape [B, C_in, T]
    Blocks: Conv1d -> Conv1d -> (transpose) -> LSTM -> Dense classifier
    """
    def __init__(
        self,
        in_channels: int,         # e.g., EEG channels
        num_classes: int,
        lstm_hidden: int = 128,
        lstm_layers: int = 1,
        bidirectional: bool = True,
        dropout: float = 0.5,
        pool: str = "mean",       # "mean" or "last" over the LSTM outputs
    ):
        super().__init__()
        # --- CNN feature extractor ---
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )

        # after CNN we’ll have [B, 128, T’]
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )
        lstm_out_dim = lstm_hidden * (2 if bidirectional else 1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(128, num_classes),
        )

        assert pool in ("mean", "last")
        self.pool = pool

    def forward(self, x):
        # x: [B, C_in, T]
        x = self.cnn(x)                 # [B, 128, T’]
        x = x.transpose(1, 2)           # [B, T’, 128] for LSTM (batch_first=True)
        y, _ = self.lstm(x)             # [B, T’, H]

        if self.pool == "mean":
            y = y.mean(dim=1)           # temporal mean-pool
        else:
            y = y[:, -1, :]             # last timestep

        logits = self.classifier(y)     # [B, num_classes]
        return logits


# SurroundSupp

In [17]:
task, run = task_keys[9]
l_freq = 3.0
h_freq = 35.0
tmin = -0.2
tmax = 2.4

task_dto = TaskDTO(subject=subject, task=task, run=run)
filter_params = FilterParamsDTO(l_freq=l_freq, h_freq=h_freq)
epoch_params = EpochParamsDTO(l_freq=l_freq, h_freq=h_freq, tmin=tmin, tmax=tmax)

task_model = subject_model.get_task(task_dto)

data_service.show_annotations(task_dto,filter_params)

{'PowerLineFrequency': 60,
 'TaskName': 'surroundSupp',
 'EEGChannelCount': 129,
 'EEGReference': 'Cz',
 'RecordingType': 'continuous',
 'RecordingDuration': 293.42,
 'SamplingFrequency': 500,
 'SoftwareFilters': 'n/a'}

In [18]:
raw = task_model.get_filtered_raw(filter_params)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 3 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 3.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 2.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 825 samples (1.650 s)



In [19]:
epochs, labels = task_model.get_epochs(epoch_params)
X = epochs.get_data() 
n_epochs, n_channels, n_times = X.shape
print("Epochs shape:", X.shape)
print("Labels example:", labels[:10])

Not setting metadata
64 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 64 events and 1301 original time points ...
0 bad epochs dropped
Epochs shape: (64, 128, 1301)
Labels example: ['bg1_fg0.0_stim2' 'bg1_fg0.3_stim3' 'bg1_fg0.6_stim1' 'bg1_fg1.0_stim2'
 'bg1_fg0.0_stim3' 'bg1_fg0.3_stim2' 'bg1_fg0.6_stim1' 'bg1_fg1.0_stim3'
 'bg1_fg0.0_stim2' 'bg1_fg0.3_stim3']


## foreground classcification

In [25]:
def extract_fg(label):
    m = re.search(r'fg([0-9.]+)', label)
    return float(m.group(1)) if m else None

fg_values = np.array([extract_fg(lbl) for lbl in labels])

valid_fg = {0.0, 0.3, 0.6, 1.0}
mask = np.isin(fg_values, list(valid_fg))

X_fg = X[mask]
y_fg = fg_values[mask]
labels_fg = np.array(labels)[mask]

print("Filtered epochs shape:", X_fg.shape)
print("Filtered fg labels:", y_fg[:10])

Filtered epochs shape: (64, 128, 1301)
Filtered fg labels: [0.  0.3 0.6 1.  0.  0.3 0.6 1.  0.  0.3]


In [26]:
fg_to_class = {0.0: 0, 0.3: 1, 0.6: 2, 1.0: 3}
y_class = np.array([fg_to_class[v] for v in y_fg])

print("Classification targets (y_class):", y_class[:10])
print("Mapping:", fg_to_class)

Classification targets (y_class): [0 1 2 3 0 1 2 3 0 1]
Mapping: {0.0: 0, 0.3: 1, 0.6: 2, 1.0: 3}


In [27]:
# 1. Flatten the filtered EEG data
n_epochs_fg, n_channels, n_times = X_fg.shape
X_fg_flat = X_fg.reshape(n_epochs, n_channels * n_times)

# 2. Convert to PyTorch tensors
# X_tensor = torch.tensor(X_fg_flat, dtype=torch.float32)
# y_tensor = torch.tensor(y_class, dtype=torch.long)

# 3. Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X_fg_flat, y_class, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

input_dim = X_fg_flat.shape[1]
num_classes = len(np.unique(y_class))
model = SimpleNN(input_dim, num_classes)

print(input_dim,num_classes)

166528 4


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs_num = 10

for epoch in range(epochs_num):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")


Epoch 1 loss: 1.3986
Epoch 2 loss: 1.3938
Epoch 3 loss: 1.3915
Epoch 4 loss: 1.3909
Epoch 5 loss: 1.3961
Epoch 6 loss: 1.3879
Epoch 7 loss: 1.3847
Epoch 8 loss: 1.3845
Epoch 9 loss: 1.3818
Epoch 10 loss: 1.3821


In [29]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb)
        pred = torch.argmax(out, dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print("Test accuracy:", correct / total)


Test accuracy: 0.23076923076923078


# Contrast Change Detection classcification